# PA-42 — Détection du genre (et âge) par locuteur
**Sprint 5 | Équipe P07

Ce notebook implémente la détection du genre et de l'âge de chaque locuteur.

**Modèle utilisé :** `audeering/wav2vec2-large-robust-6-ft-age-gender`
- Basé sur wav2vec2, entraîné sur 4 datasets de parole
- Prédit simultanément : **genre** (male/female) et **âge** (estimation en années)
- Compatible Colab CPU/GPU

**Approche :** par locuteur — on agrège tous les segments d'un SPEAKER
et on calcule le genre dominant + l'âge moyen.

**Prérequis :** pyannote déjà exécuté (diarization_segments disponible)
ou upload d'un audio + diarisation dans ce notebook.

## 1. Installation
⚠️ Redémarrer le runtime après cette cellule.

In [1]:
!pip install -q transformers torchaudio pyannote.audio silero-vad faster-whisper soundfile pydub
!apt-get install -q ffmpeg
print('✅ Installation OK — redémarre le runtime !')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 887.4 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 893.7/893.7 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 41.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.4/35.4 MB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.5/39.5 MB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 52.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/

## 2. Chargement des modèles

In [7]:
import torch, torchaudio, numpy as np, os, json, datetime
import soundfile as sf
import IPython.display as ipd
from transformers import Wav2Vec2Processor, Wav2Vec2Model
from pydub import AudioSegment
from google.colab import userdata, files

# Patch compatibilité torchaudio / pyannote
if not hasattr(torchaudio, 'AudioMetaData'):
    try:
        from torchaudio._backend.utils import AudioMetaData
    except:
        import collections
        AudioMetaData = collections.namedtuple('AudioMetaData',
            ['sample_rate', 'num_frames', 'num_channels', 'bits_per_sample', 'encoding'])
    torchaudio.AudioMetaData = AudioMetaData

from pyannote.audio import Pipeline
HF_TOKEN = userdata.get('HF_TOKEN')
diarization_pipeline = Pipeline.from_pretrained(
    'pyannote/speaker-diarization-3.1', token=HF_TOKEN)
print('✅ pyannote chargé')

# Silero VAD
vad_model, utils = torch.hub.load(
    repo_or_dir='snakers4/silero-vad', model='silero_vad',
    force_reload=False, trust_repo=True)
(get_speech_timestamps, *_) = utils
print('✅ Silero VAD chargé')

# Modèle genre + âge audeering
from transformers import AutoProcessor, AutoModelForAudioClassification

MODEL_NAME = 'audeering/wav2vec2-large-robust-6-ft-age-gender'
processor        = AutoProcessor.from_pretrained(MODEL_NAME)
age_gender_model = AutoModelForAudioClassification.from_pretrained(MODEL_NAME)
age_gender_model.eval()
print('✅ audeering wav2vec2 genre+âge chargé')

✅ pyannote chargé


Using cache found in /root/.cache/torch/hub/snakers4_silero-vad_master


✅ Silero VAD chargé


Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

[transformers] Wav2Vec2ForSequenceClassification LOAD REPORT from: audeering/wav2vec2-large-robust-6-ft-age-gender
Key                    | Status     | 
-----------------------+------------+-
age.out_proj.bias      | UNEXPECTED | 
gender.out_proj.weight | UNEXPECTED | 
gender.dense.weight    | UNEXPECTED | 
age.out_proj.weight    | UNEXPECTED | 
gender.out_proj.bias   | UNEXPECTED | 
age.dense.weight       | UNEXPECTED | 
age.dense.bias         | UNEXPECTED | 
gender.dense.bias      | UNEXPECTED | 
projector.bias         | MISSING    | 
classifier.bias        | MISSING    | 
projector.weight       | MISSING    | 
classifier.weight      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ audeering wav2vec2 genre+âge chargé


## 3. Upload et conversion audio

In [4]:
print('📂 Upload votre fichier audio...')
uploaded = files.upload()
filename = list(uploaded.keys())[0]

audio = AudioSegment.from_file(filename)
audio = audio.set_channels(1).set_frame_rate(SAMPLE_RATE)
AUDIO_PATH = 'pa42_audio/audio.wav'
audio.export(AUDIO_PATH, format='wav')
duration = len(audio) / 1000

print(f'✅ Converti → {AUDIO_PATH} ({duration:.1f}s)')
print('\n🎧 Écoute :')
ipd.display(ipd.Audio(AUDIO_PATH, rate=SAMPLE_RATE))

Output hidden; open in https://colab.research.google.com to view.

## 4. VAD + Diarisation

In [5]:
waveform, sr = torchaudio.load(AUDIO_PATH)
audio_1d    = waveform.squeeze(0)
audio_array = audio_1d.numpy()

# VAD
speech_timestamps = get_speech_timestamps(
    audio_1d, vad_model, sampling_rate=SAMPLE_RATE,
    threshold=0.4, min_speech_duration_ms=200, min_silence_duration_ms=100)
print(f'🔊 VAD : {len(speech_timestamps)} segment(s)')

# Diarisation
diarization = diarization_pipeline(AUDIO_PATH)
annotation  = diarization.speaker_diarization
diarization_segments = []
for segment, track, speaker in annotation.itertracks(yield_label=True):
    diarization_segments.append({
        'start':   round(segment.start, 3),
        'end':     round(segment.end,   3),
        'speaker': speaker
    })
speakers_found = list(set(s['speaker'] for s in diarization_segments))
print(f'🎙️ Diarisation : {len(speakers_found)} locuteur(s) → {speakers_found}')
for seg in diarization_segments:
    print(f'  [{seg["start"]:6.2f}s → {seg["end"]:6.2f}s]  {seg["speaker"]}')

🔊 VAD : 12 segment(s)
🎙️ Diarisation : 3 locuteur(s) → ['SPEAKER_01', 'SPEAKER_02', 'SPEAKER_00']
  [  0.55s →   3.49s]  SPEAKER_01
  [  3.96s →   5.57s]  SPEAKER_01
  [  5.89s →  10.70s]  SPEAKER_01
  [ 11.00s →  11.98s]  SPEAKER_01
  [ 11.98s →  22.22s]  SPEAKER_02
  [ 23.27s →  30.04s]  SPEAKER_00
  [ 30.34s →  34.39s]  SPEAKER_00
  [ 35.18s →  37.02s]  SPEAKER_01
  [ 37.49s →  40.68s]  SPEAKER_01
  [ 40.90s →  44.43s]  SPEAKER_01
  [ 44.75s →  44.83s]  SPEAKER_01
  [ 44.83s →  44.87s]  SPEAKER_02
  [ 44.87s →  44.99s]  SPEAKER_01
  [ 44.99s →  45.05s]  SPEAKER_02
  [ 45.05s →  45.97s]  SPEAKER_01
  [ 45.97s →  46.32s]  SPEAKER_02
  [ 46.32s →  46.49s]  SPEAKER_01
  [ 46.49s →  55.21s]  SPEAKER_02
  [ 55.97s →  59.50s]  SPEAKER_00
  [ 59.58s →  68.11s]  SPEAKER_00


## 5. Détection genre + âge par locuteur

Pour chaque SPEAKER :
1. On concatène tous ses segments audio
2. On passe le signal dans le modèle audeering
3. On récupère **genre** (male/female) et **âge** estimé

In [15]:
import torch
import torch.nn as nn
import numpy as np
from scipy.signal import find_peaks
from transformers import Wav2Vec2Processor, Wav2Vec2Config
from transformers.models.wav2vec2.modeling_wav2vec2 import Wav2Vec2Model
from safetensors.torch import load_file
from huggingface_hub import hf_hub_download

# ── Architecture custom audeering ──
class ModelHead(nn.Module):
    def __init__(self, config, num_labels):
        super().__init__()
        self.dense    = nn.Linear(config.hidden_size, config.hidden_size)
        self.dropout  = nn.Dropout(config.final_dropout)
        self.out_proj = nn.Linear(config.hidden_size, num_labels)
    def forward(self, features):
        x = self.dropout(features)
        x = self.dense(x)
        x = torch.tanh(x)
        x = self.dropout(x)
        return self.out_proj(x)

class AgeGenderModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.wav2vec2 = Wav2Vec2Model(config)
        self.age      = ModelHead(config, 1)
        self.gender   = ModelHead(config, 3)
    def forward(self, input_values):
        hidden = self.wav2vec2(input_values)[0]
        hidden = torch.mean(hidden, dim=1)
        return hidden, self.age(hidden), self.gender(hidden)

# ── Chargement modèle audeering ──
MODEL_NAME   = 'audeering/wav2vec2-large-robust-6-ft-age-gender'
device       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
processor    = Wav2Vec2Processor.from_pretrained(MODEL_NAME)
config       = Wav2Vec2Config.from_pretrained(MODEL_NAME)
model        = AgeGenderModel(config)
weights_path = hf_hub_download(repo_id=MODEL_NAME, filename='model.safetensors')
state_dict   = load_file(weights_path)
model.load_state_dict(state_dict, strict=False)
model        = model.to(device)
model.eval()
print(f'✅ AgeGenderModel chargé ({device})')

# ── Estimation F0 via autocorrélation (sans librosa/numba) ──
def estimate_f0(audio_array, sample_rate=16000):
    frame_size = int(0.025 * sample_rate)
    hop_size   = int(0.010 * sample_rate)
    min_period = int(sample_rate / 400)
    max_period = int(sample_rate / 50)
    f0_values  = []
    for start in range(0, len(audio_array) - frame_size, hop_size):
        frame = audio_array[start:start + frame_size]
        frame = frame - np.mean(frame)
        if np.max(np.abs(frame)) < 0.01:
            continue
        corr = np.correlate(frame, frame, mode='full')
        corr = corr[len(corr)//2:]
        peaks, _ = find_peaks(corr[min_period:max_period])
        if len(peaks) > 0:
            best_period = peaks[np.argmax(corr[min_period:max_period][peaks])] + min_period
            f0 = sample_rate / best_period
            if 50 < f0 < 400:
                f0_values.append(f0)
    return np.array(f0_values)

# ── Fonction de prédiction combinée ──
# Genre  → pitch F0 (fiable en français, language-agnostic)
# Âge    → audeering wav2vec2
def process_func(audio_array, sample_rate=16000):
    # Genre via F0
    f0_values = estimate_f0(audio_array, sample_rate)
    if len(f0_values) > 0:
        mean_f0    = float(np.mean(f0_values))
        gender     = 'male' if mean_f0 < 160 else 'female'
        confidence = min(abs(mean_f0 - 160) / 80, 1.0)
    else:
        gender, confidence, mean_f0 = 'unknown', 0.0, 0.0

    # Âge via audeering
    y = processor(audio_array, sampling_rate=sample_rate)['input_values'][0]
    y = torch.from_numpy(y.reshape(1, -1)).to(device)
    with torch.no_grad():
        _, logits_age, _ = model(y)
    age = float(logits_age[0][0]) * 100

    return age, gender, confidence, mean_f0

# ── Analyse par locuteur ──
gender_results = {}
print('🔍 Analyse genre + âge par locuteur...\n')

for spk in speakers_found:
    chunks = []
    total_duration = 0
    for seg in diarization_segments:
        if seg['speaker'] == spk:
            start = int(seg['start'] * SAMPLE_RATE)
            end   = int(seg['end']   * SAMPLE_RATE)
            chunk = audio_array[start:end]
            if len(chunk) > SAMPLE_RATE * 0.3:
                chunks.append(chunk)
                total_duration += len(chunk) / SAMPLE_RATE

    if not chunks:
        print(f'  ⚠️ {spk} — pas assez de signal')
        continue

    full_audio = np.concatenate(chunks)
    spk_path   = f'pa42_audio/spk_{spk}.wav'
    sf.write(spk_path, full_audio, SAMPLE_RATE)

    age_pred, gender, confidence, mean_f0 = process_func(full_audio)

    gender_results[spk] = {
        'gender':         gender,
        'confidence':     round(confidence, 4),
        'pitch_f0_mean':  round(mean_f0, 1),
        'age_estimate':   round(age_pred, 1),
        'total_speech_s': round(total_duration, 2)
    }

    icon = '👩' if gender == 'female' else '👨'
    print(f'  {icon} {spk} → {gender.upper()} (F0 moy: {mean_f0:.0f}Hz | confiance: {confidence:.3f}) | âge estimé: {age_pred:.0f} ans | parole: {total_duration:.1f}s')
    ipd.display(ipd.Audio(spk_path, rate=SAMPLE_RATE))
    print()

print('✅ Analyse terminée')

Output hidden; open in https://colab.research.google.com to view.

## 6. Visualisation

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Timeline colorée par genre
gender_colors = {'female': '#e74c3c', 'male': '#3498db', 'unknown': '#95a5a6'}
colors_map = plt.cm.Set2.colors

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6))

# AX1 — Timeline diarisation colorée par genre
for seg in diarization_segments:
    spk    = seg['speaker']
    gender = gender_results.get(spk, {}).get('gender', 'unknown')
    color  = gender_colors[gender]
    ax1.barh(y=spk, width=seg['end']-seg['start'],
             left=seg['start'], color=color, edgecolor='white', height=0.4)

patches = [
    mpatches.Patch(color='#e74c3c', label='Female 👩'),
    mpatches.Patch(color='#3498db', label='Male 👨')
]
ax1.legend(handles=patches, loc='upper right')
ax1.set_title('Timeline — locuteurs colorés par genre')
ax1.set_xlim(0, duration)
ax1.grid(axis='x', linestyle='--', alpha=0.4)

# AX2 — Barplot scores genre par locuteur
spk_labels = list(gender_results.keys())
female_scores = [gender_results[s]['female_score'] for s in spk_labels]
male_scores   = [gender_results[s]['male_score']   for s in spk_labels]
x = range(len(spk_labels))
ax2.bar([i-0.2 for i in x], female_scores, width=0.4, color='#e74c3c', label='Female score')
ax2.bar([i+0.2 for i in x], male_scores,   width=0.4, color='#3498db', label='Male score')
ax2.set_xticks(list(x))
ax2.set_xticklabels(spk_labels)
ax2.set_ylabel('Score')
ax2.set_title('Scores genre par locuteur')
ax2.legend()
ax2.set_ylim(0, 1)
ax2.grid(axis='y', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig('pa42_audio/pa42_gender_timeline.png', dpi=150)
plt.show()
print('✅ Visualisation sauvegardée → pa42_audio/pa42_gender_timeline.png')

## 7. Export JSON

In [ ]:
output = {
    'metadata': {
        'story':            'PA-42',
        'audio_file':       'audio.wav',
        'duration_s':       round(duration, 3),
        'processed_at':     datetime.datetime.now().isoformat(),
        'model_genre_age':  'audeering/wav2vec2-large-robust-6-ft-age-gender',
        'diarization_model':'pyannote/speaker-diarization-3.1',
        'speakers_detected': speakers_found
    },
    'genre_age_par_locuteur': gender_results,
    'diarization_segments':   diarization_segments
}

json_path = 'pa42_audio/pa42_output.json'
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print('📄 JSON exporté :')
print(json.dumps(output, ensure_ascii=False, indent=2))

## 8. Récapitulatif PA-42

In [ ]:
print('=' * 55)
print('  RÉCAPITULATIF — PA-42 Détection Genre + Âge')
print('=' * 55)
print(f'  Audio      : audio.wav ({duration:.1f}s)')
print(f'  Locuteurs  : {len(speakers_found)}')
print()
for spk, res in gender_results.items():
    icon = '👩' if res['gender'] == 'female' else '👨'
    print(f'  {icon} {spk}')
    print(f'     Genre    : {res["gender"].upper()} (confiance: {res["confidence"]:.3f})')
    print(f'     Âge est. : {res["age_estimate"]:.0f} ans')
    print(f'     Parole   : {res["total_speech_s"]}s')
    print()
print('=' * 55)